# Partner Hub (Jupyter-first)

We work here so we can see what works and reproduce it. No demos.

Workflow:
1) Environment + last-week git truth
2) Load latest sessions/ save
3) Run real entrypoints
4) Earnings Reaction Study (MU-style)


In [3]:
import json, subprocess
from pathlib import Path
import pandas as pd

REPO_ROOT = Path('/workspaces/quantum-ai-trader_v1.1').resolve()
SESSIONS_DIR = REPO_ROOT / 'sessions'
CACHE_DIR = REPO_ROOT / 'data' / 'earnings_event_study_cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)

def sh(cmd: str) -> str:
    p = subprocess.run(cmd, shell=True, cwd=str(REPO_ROOT), capture_output=True, text=True)
    return ((p.stdout or '') + (p.stderr or '')).strip()

print('Repo:', REPO_ROOT)
print('Python:', sh('python -V') or 'unknown')
print('Git branch:', sh('git rev-parse --abbrev-ref HEAD') or 'unknown')


Repo: /workspaces/quantum-ai-trader_v1.1
Python: Python 3.12.1
Git branch: main


In [4]:
print(sh('git --no-pager log --since="7 days ago" --date=short --pretty=format:"%h %ad %s" | head -n 30'))


405a7ce 2025-12-17 2 WEEKS - Prove worth or done
38e3b32 2025-12-17 FINAL WARNING - Last chance tomorrow or legacy model
7900676 2025-12-17 Real state - KDK only, system needs work
823c6e4 2025-12-17 📌 PICKUP_HERE.md - continuation point
cf0ee25 2025-12-17 📓 Final notebook save with restore cell
84b9ad9 2025-12-17 💾 FULL SESSION SAVE - Dec 17, 2025
9644498 2025-12-17 🌙 Final session save with goodnight summary
bedc2e3 2025-12-17 💾 SESSION SAVE: Dec 17, 2025 - Complete Discovery Analysis
727fbbc 2025-12-17 🔬 BRUTAL TRUTH: After DeepSeek & Claude destroyed our methodology
16388c4 2025-12-17 🎯 MASSIVE SESSION: 52 experiments, 65+ edges, 10 paper trades queued
909897e 2025-12-16 SESSION DEC 16: Found RSI<10 = 80.8% WR edge, tested 12 strategies, saved handoff for tomorrow
da71cf9 2025-12-15 UPDATED RESUME PROTOCOL: Explicit instructions for tomorrow's Copilot to read ALL 6 key files in order, process everything, and respond with complete context. No confusion, no memory loss, seamless cont

In [5]:
def find_latest_session_folder(base: Path) -> Path | None:
    if not base.exists():
        return None
    candidates = [p for p in base.iterdir() if p.is_dir()]
    if not candidates:
        return None

    def score(p: Path) -> tuple[int, float]:
        want = [
            p / 'SESSION_SUMMARY.json',
            p / 'PARAMETERS.json',
            p / 'ALPACA_STATE.json',
            p / 'BEST_2Y.csv',
            p / 'SIGNALS_DF.csv',
            p / 'BACKTEST_DF.csv',
        ]
        hits = sum(1 for f in want if f.exists())
        return (hits, p.stat().st_mtime)

    return sorted(candidates, key=score, reverse=True)[0]

LATEST_SESSION = find_latest_session_folder(SESSIONS_DIR)
print('Latest session folder:', LATEST_SESSION)
if LATEST_SESSION:
    files = sorted([p.name for p in LATEST_SESSION.iterdir() if p.is_file()])
    print('Files:', len(files))
    print('Preview:')
    print('  ' + '\n  '.join(files[:40]))


Latest session folder: /workspaces/quantum-ai-trader_v1.1/sessions/20251217_0525_FULL_SAVE
Files: 22
Preview:
  ALPACA_STATE.json
  ATOMIC_TICKERS.json
  ATOMIC_TRADES.csv
  BACKTEST_DF.csv
  BEST_2Y.csv
  BLUE_CHIPS.json
  CONTINUE_TOMORROW.md
  FEAR_ANALYSIS.csv
  FEATURE_COLS.json
  MACRO_DF.csv
  NUCLEAR_SYSTEM.pkl
  PARAMETERS.json
  PRODUCTION_GENERATOR.pkl
  QUALITY_STOCKS.json
  REGIME_SIGNALS_DF.csv
  RISK_METRICS.csv
  SESSION_SUMMARY.json
  SIGNALS_DF.csv
  WATCHLIST.json
  XGBOOST_MODEL.pkl
  YOUR_TICKERS.json
  YOUR_TRADES.csv


In [6]:
def load_json(path: Path):
    if not path.exists():
        return None
    return json.loads(path.read_text())

def load_csv(path: Path) -> pd.DataFrame | None:
    if not path.exists():
        return None
    return pd.read_csv(path)

session_summary = load_json(LATEST_SESSION / 'SESSION_SUMMARY.json') if LATEST_SESSION else None
parameters = load_json(LATEST_SESSION / 'PARAMETERS.json') if LATEST_SESSION else None
alpaca_state = load_json(LATEST_SESSION / 'ALPACA_STATE.json') if LATEST_SESSION else None

print('SESSION_SUMMARY.json:', 'loaded' if session_summary else 'missing')
print('PARAMETERS.json:', 'loaded' if parameters else 'missing')
print('ALPACA_STATE.json:', 'loaded' if alpaca_state else 'missing')


SESSION_SUMMARY.json: loaded
PARAMETERS.json: loaded
ALPACA_STATE.json: loaded


In [7]:
tables = {}
if LATEST_SESSION:
    for name in ['BEST_2Y.csv', 'SIGNALS_DF.csv', 'BACKTEST_DF.csv', 'YOUR_TRADES.csv', 'RISK_METRICS.csv']:
        p = LATEST_SESSION / name
        if p.exists():
            tables[name] = load_csv(p)
            print(f'Loaded {name}: {tables[name].shape}')

if 'BEST_2Y.csv' in tables:
    display(tables['BEST_2Y.csv'].head(10))


Loaded BEST_2Y.csv: (55, 9)
Loaded SIGNALS_DF.csv: (3447, 5)
Loaded BACKTEST_DF.csv: (926, 8)
Loaded YOUR_TRADES.csv: (27, 9)
Loaded RISK_METRICS.csv: (5, 7)


,ticker,date,rsi,vix,result,return,days,year,return_pct
0,AAPL,2024-08-05,18.134256,38.570000,WIN,0.05,1,2024,5.0
1,MSFT,2024-08-05,18.575863,38.570000,WIN,0.05,2,2024,5.0
2,AMZN,2025-02-25,17.930745,19.430000,LOSS,-0.02,3,2025,-2.0
3,AMZN,2025-03-04,15.687199,23.510000,LOSS,-0.02,3,2025,-2.0
4,AMZN,2025-11-21,19.479574,23.430000,WIN,0.05,1,2025,5.0
5,META,2025-11-07,17.938028,19.080000,LOSS,-0.02,4,2025,-2.0
6,META,2025-11-14,12.348191,19.830000,LOSS,-0.02,2,2025,-2.0
7,META,2025-11-18,11.241453,24.690001,WIN,0.05,5,2025,5.0
8,TSLA,2025-03-07,17.251324,23.370001,LOSS,-0.02,1,2025,-2.0
9,TSLA,2025-03-11,14.490222,26.920000,WIN,0.05,1,2025,5.0


## Run the real entrypoints
These call actual scripts in the repo.


In [8]:
print(sh('python run_research_pipeline.py'))


Note: pyts not installed. Install with: pip install pyts
Note: PySR not installed. Install with: pip install pysr
Note: DEAP not installed. Install with: pip install deap
Note: stable-baselines3 not installed. Install with: pip install stable-baselines3
Note: gymnasium not installed. Install with: pip install gymnasium

🚀 GOLDEN ARCHITECTURE - LOCAL RESEARCH PIPELINE

[1/5] Loading Data...
✓ Loaded 250 candles for SPY
Columns: MultiIndex([( 'Close', 'SPY'),
            (  'High', 'SPY'),
            (   'Low', 'SPY'),
            (  'Open', 'SPY'),
            ('Volume', 'SPY')],
           names=['Price', 'Ticker'])
Detected MultiIndex columns, flattening...
New Columns: Index(['Close', 'High', 'Low', 'Open', 'Volume'], dtype='object', name='Price')

[2/5] Running Vision Engine (GASF)...
✓ Vision Analysis Complete
  Dominant Pattern: double_top
  Confidence: 11.1%

[3/5] Running Logic Engine (Symbolic Regression)...
✓ Logic Discovery Complete
  Equation: + 0.099 * returns + 0.034 * rs

In [9]:
print(sh('python daily_scanner.py'))


🌅 MORNING SCAN - 2025-12-17 22:45

🚨 ALERTS:
⚠️ IONQ: News 2.5x
⚠️ RGTI: RSI 37
⚠️ QBTS: News 3.0x
⚠️ LEU: RSI 35
⚠️ OKLO: RSI 35
⚠️ UUUU: RSI 39
⚠️ SMR: RSI 32
⚠️ TLRY: News 22.0x
⚠️ WULF: News 3.0x
⚠️ SMCI: News 3.2x
⚠️ AMD: News 2.3x
⚠️ COIN: News 2.3x
⚠️ TSLA: News 3.2x


## Earnings Reaction Study (MU-style)
We test if positive earnings tends to produce predictable gains.

Definitions:
- Positive earnings: EPS reported ≥ EPS estimate (when available)
- Returns measured from previous trading day close:
  - Gap: prev close → event day open
  - Day1: prev close → event day close
  - Drift: prev close → close after N trading days


In [10]:
import yfinance as yf
from datetime import timedelta

def _flatten_yf(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or len(df) == 0:
        return df
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    return df

def get_earnings_events(ticker: str, limit: int = 12) -> pd.DataFrame:
    t = yf.Ticker(ticker)
    ed = t.get_earnings_dates(limit=limit)
    if ed is None or len(ed) == 0:
        raise RuntimeError(f'No earnings dates returned for {ticker}')
    ed = ed.reset_index().rename(columns={'Earnings Date': 'earnings_datetime'})
    ed['earnings_date'] = pd.to_datetime(ed['earnings_datetime']).dt.date
    return ed

def get_daily_prices(ticker: str, start, end) -> pd.DataFrame:
    df = yf.download(ticker, start=start, end=end, progress=False, auto_adjust=False)
    df = _flatten_yf(df)
    if df is None or len(df) == 0:
        raise RuntimeError(f'No daily prices for {ticker} from {start} to {end}')
    df = df.copy()
    df.index = pd.to_datetime(df.index).date
    return df

def earnings_event_study(ticker: str, limit: int = 12, horizon_days: int = 5, use_cache: bool = True) -> pd.DataFrame:
    cache_path = CACHE_DIR / f'{ticker}_earnings_event_study_h{horizon_days}.csv'
    if use_cache and cache_path.exists():
        return pd.read_csv(cache_path)

    events = get_earnings_events(ticker, limit=limit)
    min_dt = pd.to_datetime(events['earnings_datetime']).min().date() - timedelta(days=10)
    max_dt = pd.to_datetime(events['earnings_datetime']).max().date() + timedelta(days=30)
    px = get_daily_prices(ticker, start=min_dt.isoformat(), end=max_dt.isoformat())
    trading_days = list(px.index)

    def prev_day(d):
        prevs = [x for x in trading_days if x < d]
        return prevs[-1] if prevs else None

    def next_day(d):
        nexts = [x for x in trading_days if x >= d]
        return nexts[0] if nexts else None

    def plus_n(d, n):
        if d not in trading_days:
            return None
        i = trading_days.index(d)
        j = i + n
        return trading_days[j] if j < len(trading_days) else None

    rows = []
    for _, r in events.iterrows():
        edate = r['earnings_date']
        prev = prev_day(edate)
        evt = next_day(edate)
        if not prev or not evt:
            continue
        evt_plus = plus_n(evt, horizon_days)
        if not evt_plus:
            continue

        prev_close = float(px.loc[prev, 'Close'])
        evt_open = float(px.loc[evt, 'Open'])
        evt_close = float(px.loc[evt, 'Close'])
        h_close = float(px.loc[evt_plus, 'Close'])

        gap = (evt_open / prev_close) - 1.0
        day1 = (evt_close / prev_close) - 1.0
        drift = (h_close / prev_close) - 1.0

        eps_est = r.get('EPS Estimate', None)
        eps_act = r.get('Reported EPS', None)
        is_pos = None
        surprise = None
        try:
            if pd.notna(eps_est) and pd.notna(eps_act):
                eps_est_f = float(eps_est)
                eps_act_f = float(eps_act)
                surprise = eps_act_f - eps_est_f
                is_pos = surprise >= 0
        except Exception:
            pass

        rows.append({
            'ticker': ticker,
            'earnings_datetime': str(r['earnings_datetime']),
            'earnings_date': str(edate),
            'prev_trading_day': str(prev),
            'event_trading_day': str(evt),
            f'event_plus_{horizon_days}d': str(evt_plus),
            'gap_prevclose_to_open': gap,
            'ret_prevclose_to_close_1d': day1,
            f'ret_prevclose_to_close_{horizon_days}d': drift,
            'eps_estimate': eps_est,
            'eps_reported': eps_act,
            'eps_surprise': surprise,
            'positive_eps_surprise': is_pos,
        })

    out = pd.DataFrame(rows)
    out.to_csv(cache_path, index=False)
    return out


In [11]:
TICKERS = ['MU', 'NVDA', 'PLTR', 'HOOD', 'AVGO']
LIMIT = 12
HORIZON_DAYS = 5

frames = []
for t in TICKERS:
    try:
        df = earnings_event_study(t, limit=LIMIT, horizon_days=HORIZON_DAYS, use_cache=True)
        print(f'{t}: {len(df)} events')
        frames.append(df)
    except Exception as e:
        print(f'{t}: failed -> {e}')

events = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
display(events.head(25))

have_eps = events[events['positive_eps_surprise'].notna()].copy() if len(events) else pd.DataFrame()
if len(have_eps) == 0:
    print('No EPS estimate/reported fields available from yfinance for these events (this can happen).')
else:
    cols = ['gap_prevclose_to_open', 'ret_prevclose_to_close_1d', f'ret_prevclose_to_close_{HORIZON_DAYS}d']
    summary = have_eps.groupby('positive_eps_surprise')[cols].agg(['count', 'mean', 'median'])
    display(summary)


MU: 25 events
NVDA: 24 events
PLTR: 21 events
HOOD: 18 events
AVGO: 23 events


,ticker,earnings_datetime,earnings_date,prev_trading_day,event_trading_day,event_plus_5d,gap_prevclose_to_open,ret_prevclose_to_close_1d,ret_prevclose_to_close_5d,eps_estimate,eps_reported,eps_surprise,positive_eps_surprise
0,MU,2025-09-23 16:00:00-04:00,2025-09-23,2025-09-22,2025-09-23,2025-09-30,0.006439,0.010874,0.016401,2.69,2.83,0.14,True
1,MU,2025-06-25 16:00:00-04:00,2025-06-25,2025-06-24,2025-06-25,2025-07-02,-0.010163,-0.005160,-0.048237,1.59,1.91,0.32,True
2,MU,2025-03-20 16:00:00-04:00,2025-03-20,2025-03-19,2025-03-20,2025-03-27,-0.004703,0.009210,-0.106800,1.42,1.56,0.14,True
3,MU,2024-12-18 16:00:00-05:00,2024-12-18,2024-12-17,2024-12-18,2024-12-26,0.015193,-0.043278,-0.172928,1.77,1.79,0.02,True
4,MU,2024-09-25 16:00:00-04:00,2024-09-25,2024-09-24,2024-09-25,2024-10-02,0.001702,0.018830,0.062234,1.11,1.18,0.07,True
5,MU,2024-06-26 16:00:00-04:00,2024-06-26,2024-06-25,2024-06-26,2024-07-03,0.013605,0.008787,-0.030470,0.53,0.62,0.09,True
6,MU,2024-03-20 16:00:00-04:00,2024-03-20,2024-03-19,2024-03-20,2024-03-27,0.010532,0.023936,0.268617,-0.24,0.42,0.66,True
7,MU,2023-12-20 16:00:00-05:00,2023-12-20,2023-12-19,2023-12-20,2023-12-28,-0.008641,-0.042351,0.046611,-1.01,-0.95,0.06,True
8,MU,2023-09-27 16:00:00-04:00,2023-09-27,2023-09-26,2023-09-27,2023-10-04,0.002649,0.003974,0.002502,-1.18,-1.07,0.11,True
9,MU,2023-06-28 16:00:00-04:00,2023-06-28,2023-06-27,2023-06-28,2023-07-06,-0.017068,0.004192,-0.083246,-1.57,-1.43,0.14,True


gap_prevclose_to_open                      \
                                      count      mean    median   
positive_eps_surprise                                             
False                                    12 -0.044211 -0.011509   
True                                     99  0.000473  0.001702   

                      ret_prevclose_to_close_1d                      \
                                          count      mean    median   
positive_eps_surprise                                                 
False                                        12 -0.065557 -0.054913   
True                                         99  0.003256 -0.002068   

                      ret_prevclose_to_close_5d                      
                                          count      mean    median  
positive_eps_surprise                                                
False                                        12 -0.031349 -0.004706  
True                                         99  0.034605  0.022551

## Scale: Earnings Study Across a Larger Universe
This runs the same event study for a larger ticker set sourced from repo artifacts (sessions + watchlists) and summarizes results where EPS fields exist.

In [12]:
import re

def load_tickers_from_txt(path: Path) -> list[str]:
    if not path.exists():
        return []
    tickers: list[str] = []
    for line in path.read_text().splitlines():
        s = line.strip().upper()
        if not s or s.startswith('#'):
            continue
        s = re.sub(r'[^A-Z0-9\.\-]', '', s)
        if s:
            tickers.append(s)
    return tickers

universe = set()

# 1) From the latest session's BEST_2Y.csv (if present)
if 'BEST_2Y.csv' in tables and 'ticker' in tables['BEST_2Y.csv'].columns:
    universe |= set(tables['BEST_2Y.csv']['ticker'].astype(str).str.upper().unique())

# 2) From repo watchlists
for wl in ['alpha_76_watchlist.txt', 'watchlist.txt', 'merged_watchlist.txt', 'small_caps_watchlist.txt']:
    universe |= set(load_tickers_from_txt(REPO_ROOT / wl))

universe = sorted([t for t in universe if t and t != 'NAN'])
print('Universe tickers:', len(universe))
print('Preview:', universe[:40])

MAX_TICKERS = 50   # raise if you want (rate limits apply)
LIMIT_EVENTS = 8  # earnings events per ticker
HORIZON_DAYS = 5

frames_universe: list[pd.DataFrame] = []
failures: list[tuple[str, str]] = []

for i, t in enumerate(universe[:MAX_TICKERS], start=1):
    try:
        df_t = earnings_event_study(t, limit=LIMIT_EVENTS, horizon_days=HORIZON_DAYS, use_cache=True)
        frames_universe.append(df_t)
        if i % 10 == 0:
            print(f'... {i}/{min(MAX_TICKERS, len(universe))} tickers done')
    except Exception as e:
        failures.append((t, str(e)))

if failures:
    print('Failures:', len(failures))
    print('First 10 failures:', failures[:10])

events_universe = pd.concat(frames_universe, ignore_index=True) if frames_universe else pd.DataFrame()
print('Universe events rows:', len(events_universe))

if len(events_universe):
    have_eps_universe = events_universe[events_universe['positive_eps_surprise'].notna()].copy()
    print('Events with EPS estimate+reported:', len(have_eps_universe))

    if len(have_eps_universe) == 0:
        print('No EPS estimate/reported fields available in this run (can happen with yfinance).')
    else:
        cols_u = ['gap_prevclose_to_open', 'ret_prevclose_to_close_1d', f'ret_prevclose_to_close_{HORIZON_DAYS}d']
        summary_universe = have_eps_universe.groupby('positive_eps_surprise')[cols_u].agg(['count', 'mean', 'median'])
        display(summary_universe)
        
        coverage = have_eps_universe['positive_eps_surprise'].value_counts(dropna=False)
        print('positive_eps_surprise counts:')
        print(coverage)
else:
    print('No events collected. Consider lowering MAX_TICKERS, or check yfinance connectivity.')


Universe tickers: 173
Preview: ['AAPL', 'ABBV', 'ACHR', 'ADBE', 'AEVA', 'AFRM', 'AI', 'AKRO', 'AKYA', 'ALKT', 'AMBA', 'AMD', 'AMPL', 'AMSC', 'AMZN', 'APLS', 'APP', 'ARRY', 'ASTS', 'AVGO', 'AXNX', 'AXP', 'BA', 'BAC', 'BBAI', 'BE', 'BEAM', 'BKSY', 'BLDP', 'BLK', 'C', 'CAT', 'CELH', 'CHPT', 'CLSK', 'COIN', 'COP', 'CRM', 'CRNC', 'CRSP']


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: AKYA"}}}

1 Failed download:
['AKYA']: YFTzMissingError('possibly delisted; no timezone found')


... 10/50 tickers done
... 20/50 tickers done



1 Failed download:
['AXNX']: YFTzMissingError('possibly delisted; no timezone found')


... 30/50 tickers done
... 40/50 tickers done


DIA: No earnings dates found, symbol may be delisted


... 50/50 tickers done
Failures: 3
First 10 failures: [('AKYA', 'No daily prices for AKYA from 2021-05-08 to 2025-09-03'), ('AXNX', 'No daily prices for AXNX from 2018-12-01 to 2024-12-07'), ('DIA', 'No earnings dates returned for DIA')]
Universe events rows: 1039
Events with EPS estimate+reported: 1026


gap_prevclose_to_open                      \
                                      count      mean    median   
positive_eps_surprise                                             
False                                   268 -0.005200 -0.002186   
True                                    758  0.003386  0.002629   

                      ret_prevclose_to_close_1d                      \
                                          count      mean    median   
positive_eps_surprise                                                 
False                                       268 -0.014739 -0.013167   
True                                        758  0.005149  0.003268   

                      ret_prevclose_to_close_5d                      
                                          count      mean    median  
positive_eps_surprise                                                
False                                       268 -0.004085 -0.007877  
True                                        758  0.022584  0.008424

positive_eps_surprise counts:
positive_eps_surprise
True     758
False    268
Name: count, dtype: int64


## Monthly Event Refresh (Artifacts)
Run the event modules from terminal (recommended for reliability and speed), then use the cells below to load and display the latest outputs.

Terminal commands (examples):
- `python scripts/export_earnings_calendar.py --source universe300 --max-tickers 300 --days-ahead 45`
- `python scripts/run_earnings_event_study_batch.py --source universe300 --max-tickers 300 --limit-events 8 --horizon-days 5 --benchmark SPY --use-cache`


In [ ]:
from pathlib import Path
import pandas as pd

def latest_subdir(base: Path) -> Path | None:
    if not base.exists():
        return None
    dirs = [p for p in base.iterdir() if p.is_dir()]
    if not dirs:
        return None
    return sorted(dirs, key=lambda p: p.stat().st_mtime, reverse=True)[0]

def latest_subdir_with_file(base: Path, filename: str) -> Path | None:
    if not base.exists():
        return None
    dirs = [p for p in base.iterdir() if p.is_dir()]
    dirs = [p for p in dirs if (p / filename).exists()]
    if not dirs:
        return None
    return sorted(dirs, key=lambda p: p.stat().st_mtime, reverse=True)[0]

# Loads latest earnings calendar artifact created by scripts/export_earnings_calendar.py
cal_base = REPO_ROOT / 'data' / 'events' / 'earnings_calendar'
cal_run = latest_subdir_with_file(cal_base, 'earnings_calendar.csv')
print('Latest earnings calendar run:', cal_run)

if cal_run:
    cal_path = cal_run / 'earnings_calendar.csv'
    cal_df = pd.read_csv(cal_path)
    print('Rows:', len(cal_df))
    display(cal_df.head(25))
else:
    print('No calendar runs found yet. Run the exporter from terminal (see cell above).')


In [ ]:
# Loads latest earnings event study artifacts created by scripts/run_earnings_event_study_batch.py
from pathlib import Path
import pandas as pd

def latest_subdir_with_file(base: Path, filename: str) -> Path | None:
    if not base.exists():
        return None
    dirs = [p for p in base.iterdir() if p.is_dir()]
    dirs = [p for p in dirs if (p / filename).exists()]
    if not dirs:
        return None
    return sorted(dirs, key=lambda p: p.stat().st_mtime, reverse=True)[0]

runs_base = REPO_ROOT / 'data' / 'earnings_event_study_runs'
run_dir = latest_subdir_with_file(runs_base, 'summary_by_eps_surprise.csv')
print('Latest completed earnings study run:', run_dir)

if run_dir:
    summary_path = run_dir / 'summary_by_eps_surprise.csv'
    failures_path = run_dir / 'failures.csv'
    events_path = run_dir / 'events.csv'

    if failures_path.exists():
        failures_df = pd.read_csv(failures_path)
        print('Failures:', len(failures_df))
        display(failures_df.head(15))
    else:
        print('Missing:', failures_path)

    summary_df = pd.read_csv(summary_path, index_col=0)
    display(summary_df)

    # Optional: peek first rows without loading huge file
    if events_path.exists():
        events_df = pd.read_csv(events_path, nrows=10)
        print('Events preview (first 10 rows):')
        display(events_df)
else:
    print('No completed study runs found yet. Wait for the batch job to finish or run a smaller batch.')
